In [4]:
force_rerun = True

# imports
import os
import sys

# code for enabling this notebook to work within cursor
coralme_dir = '/home/chris/zuniga/coralme/' #'../'
base_dir = os.path.join(coralme_dir, 'species_files', 'Pseudomonas_files')
sys.path.insert(0, coralme_dir)

import subprocess
from cobra.io import load_json_model, write_sbml_model, save_json_model
from bs4 import BeautifulSoup
import pandas as pd
import json

def extract_json_from_window_data(html):
    start = html.find("window.data =")
    if start == -1:
        raise ValueError("Could not find 'window.data =' in the HTML.")

    start += len("window.data =")
    i = start
    brace_count = 0
    in_string = False
    escape = False

    # Skip whitespace to find the first {
    while html[i] in " \n\r\t":
        i += 1

    if html[i] != '{':
        raise ValueError("Expected '{' after 'window.data ='")

    json_start = i
    brace_count += 1
    i += 1

    # Parse until all braces are closed
    while i < len(html):
        char = html[i]

        if in_string:
            if escape:
                escape = False
            elif char == '\\':
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0:
                    return html[json_start:i + 1]
        i += 1

    raise ValueError("Could not parse full JSON object from 'window.data ='")


In [5]:
# look into problem causing reactions
for f in os.listdir(os.path.join(base_dir, 'individual_species')):
    if 'Reference' in f: continue
    
    # look to see if memote solution already exists
    M_json_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model.json')
    M_json_updated_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model_fixed.json')
    if not force_rerun or not os.path.exists(M_json_updated_path):
        continue
    model = load_json_model(M_json_path)
    
    problematic_metabs = {}
    problematic_rxns = []
    for rxn in model.reactions:
        if 'EX_' in rxn.id or 'BIOMASS' in rxn.id: continue # these are imbalanced but obviously so
        input_eles = {}
        for metab in rxn.reactants:
            coef = rxn.get_coefficient(metab)
            for ele, ct in metab.elements.items():
                if ele not in input_eles:
                    input_eles.update({ele : 0})
                input_eles[ele] += abs(coef*ct)
        output_eles = {}
        for metab in rxn.products:
            coef = rxn.get_coefficient(metab)
            for ele, ct in metab.elements.items():
                if ele not in output_eles:
                    output_eles.update({ele : 0})
                output_eles[ele] += abs(coef*ct)
        if input_eles != output_eles:
            problematic_rxns.append(rxn.id)
            for metab in rxn.metabolites:
                if metab not in problematic_metabs:
                    problematic_metabs.update({metab : 0})
                problematic_metabs[metab] += 1
    print(f+': %age of problematic reactions = {:.2f}'.format(100*len(problematic_rxns) / len(model.reactions)))
    if len(problematic_rxns) < 10: print(problematic_rxns)
    print()

    # let's fix reactions
    if 'NITR_NO' in problematic_rxns:
        rxn = model.reactions.get_by_id('NITR_NO')
        rxn.add_metabolites({
            'h_c': 1.0,
        })
        # the model won't run unless you reduce the ATP requirement (was 8.4)
        model.reactions.get_by_id("ATPM").lower_bound = 5
        model.reactions.get_by_id("ATPM").upper_bound = 5
        save_json_model(model, M_json_updated_path)

CP008896: %age of problematic reactions = 0.06
['NITR_NO']

CP053697: %age of problematic reactions = 0.06
['NITR_NO']

CP026386: %age of problematic reactions = 0.06
['NITR_NO']

CP069317: %age of problematic reactions = 0.06
['NITR_NO']

AE004091.2: %age of problematic reactions = 0.05
['NITR_NO']

CP014784: %age of problematic reactions = 0.06
['NITR_NO']

CP073105: %age of problematic reactions = 0.06
['NITR_NO']

CP065866: %age of problematic reactions = 0.05
['NITR_NO']

CP061848: %age of problematic reactions = 0.06
['NITR_NO']

CP012831: %age of problematic reactions = 0.06
['NITR_NO']

CP008749.1: %age of problematic reactions = 0.05
['NITR_NO']

CP032419: %age of problematic reactions = 0.06
['NITR_NO']

CP038001: %age of problematic reactions = 0.06
['NITR_NO']

CP022560: %age of problematic reactions = 0.06
['NITR_NO']

CP076683: %age of problematic reactions = 0.06
['NITR_NO']

CP045416: %age of problematic reactions = 0.06
['NITR_NO']

AP022324: %age of problematic reacti

In [6]:
# run memote on each, saving to json
output_dfs = []
reverse = ['Presence of Gene Annotation', 'Presence of Metabolite Annotation', 'Presence of Reaction Annotation']
for f in os.listdir(os.path.join(base_dir, 'individual_species')):
    if 'Reference' in f: continue
    
    # look to see if memote solution already exists
    print(f+' loading...', end = '')
    #M_json_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model.json')
    M_json_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model_fixed.json')
    M_xml_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model.xml')
    report_path = os.path.join(base_dir, 'individual_species', f, 'outputs', 'memote_report.html')
    if force_rerun or (os.path.exists(M_json_path) and not os.path.exists(M_xml_path)):
        model = load_json_model(M_json_path)
        write_sbml_model(model, M_xml_path)

    # run memote
    if force_rerun or not os.path.exists(report_path):
        print(f+' running memote...', end = '')
        result = subprocess.run(['memote', 'report', 'snapshot', '--filename', report_path, M_xml_path], capture_output=True, text=True)

    # Load and extract results
    with open(report_path, encoding="utf-8") as file:
        html = file.read()
    
    json_str = extract_json_from_window_data(html)
    data = json.loads(json_str)
    
    # Summarize tests
    test_names = []
    test_results = []
    test_metrics = []
    
    # Loop through all tests in the memote data
    for test_id, test in data.get("tests", {}).items():
        name = test.get("title", test_id)
        result = test.get("result", None)
        metric = test.get("metric", None)
    
        # Normalize result (handle dicts like per-database results)
        if isinstance(result, dict):
            result = None
    
        # Normalize metric
        if not isinstance(metric, (int, float)):
            metric = None

        if name in reverse:
            metric = 1 - metric
    
        # Append to lists
        test_names.append(name)
        test_results.append(result)
        test_metrics.append(metric)
    
    # add to outputs
    output_df = pd.DataFrame(index = test_names)
    output_df[f+'_result'] = test_results
    output_df[f+'_metric'] = test_metrics
    output_dfs.append(output_df)
    print('done!')
    
# concatenate all these output dataframes together
output_df = pd.concat(output_dfs, axis = 1)

CP008896 loading...CP008896 running memote...done!
CP053697 loading...CP053697 running memote...done!
CP026386 loading...CP026386 running memote...done!
CP069317 loading...CP069317 running memote...done!
AE004091.2 loading...AE004091.2 running memote...done!
CP014784 loading...CP014784 running memote...done!
CP073105 loading...CP073105 running memote...done!
CP065866 loading...CP065866 running memote...done!
CP061848 loading...CP061848 running memote...done!
CP012831 loading...CP012831 running memote...done!
CP008749.1 loading...CP008749.1 running memote...done!
CP032419 loading...CP032419 running memote...done!
CP038001 loading...CP038001 running memote...done!
CP022560 loading...CP022560 running memote...done!
CP076683 loading...CP076683 running memote...done!
CP045416 loading...CP045416 running memote...done!
AP022324 loading...AP022324 running memote...done!
LS483372 loading...LS483372 running memote...done!
CP022562 loading...CP022562 running memote...done!
LR590473 loading...LR59

In [ ]:
# inspect a specific one
rxn = model.reactions.get_by_id('NITR_NO')
input_eles = {}
print(rxn.id, end = ': ')
print(rxn.reaction)
for metab in rxn.reactants:
    coef = rxn.get_coefficient(metab)
    print(str(coef)+' * '+metab.id, end = ': ')
    for ele, ct in metab.elements.items():
        if ele not in input_eles:
            input_eles.update({ele : 0})
        input_eles[ele] += abs(coef*ct)
        print(str(abs(coef*ct))+' '+ele, end = ', ')
    print()
output_eles = {}
for metab in rxn.products:
    coef = rxn.get_coefficient(metab)
    print(str(coef)+' * '+metab.id, end = ': ')
    for ele, ct in metab.elements.items():
        if ele not in output_eles:
            output_eles.update({ele : 0})
        output_eles[ele] += abs(coef*ct)
        print(str(abs(coef*ct))+' '+ele, end = ', ')
    print()
print('reactant total : %s' % input_eles)
print('product total : %s' % output_eles)

In [ ]:
# let's show the top problem causers
top_metabs = [k for k, _ in sorted(problematic_metabs.items(), key = lambda x : -x[1])]
for metab in top_metabs[0:30]:
    print(metab.id+' : n = '+str(problematic_metabs[metab]))